In [37]:
import json
import copy
import tldextract

from collections import Counter

In [38]:
"""
Infer domain-squatting transformations and express them as Hashcat-style rules.

Input JSON format:
{
  "original.com": {
     "transformed1.com": <int>,
     "transformed2.com": <int>,
     ...
  },
  ...
}

We compare SLDs (second-level domains, excluding .com), infer a minimal
Damerau-Levenshtein edit script, then map edits to hashcat rules:

Mapping:
- delete first char            ->  [
- delete last char             ->  ]
- delete at position N         ->  D N
- prepend char X               ->  ^ X   (multi-char prefix becomes multiple ^ rules,
                                         applied in reverse order to preserve prefix)
- append char X                ->  $ X   (multi-char suffix becomes multiple $ rules)
- insert char X at position N  ->  i N X
- overwrite position N with X  ->  o N X
- swap positions i and i+1     ->  * i (i+1)      (for adjacent transposition)

Rules are emitted as space-separated tokens like best64.rule.
Multiple ops are combined in order into ONE composite rule string.

Outputs:
- Writes rule_counts.json
- Writes domain.rule sorted by frequency (desc, stable tie-break)

NOTE: Domain names are lowercased; case rules are not generated.
"""

from __future__ import annotations
import json
from dataclasses import dataclass
from collections import Counter
from typing import List, Tuple, Union, Iterable, Set

# ---------------------------
# base36 encoding/decoding
# ---------------------------
_BASE36 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
_BASE36_MAP = {ch: i for i, ch in enumerate(_BASE36)}

def enc_pos(n: int) -> str:
    if n < 0 or n >= len(_BASE36):
        raise ValueError(f"Position {n} out of encodable range 0-35")
    return _BASE36[n]

def dec_pos(ch: str) -> int:
    ch = ch.upper()
    if ch not in _BASE36_MAP:
        raise ValueError(f"Bad base36 position: {ch}")
    return _BASE36_MAP[ch]

# ---------------------------
# edit-script inference
# ---------------------------
@dataclass
class Op:
    kind: str   # 'ins', 'del', 'sub', 'trans', 'keep'
    i: int      # index in original (for ins, position before which inserted)
    j: int      # index in transformed
    a: str = '' # original char(s)
    b: str = '' # transformed char(s)

def sld(domain: str) -> str:
    return domain.split('.', 1)[0].lower()

def damerau_levenshtein_ops(a: str, b: str) -> List[Op]:
    """Minimal edit script with adjacent transpositions."""
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    back = [[None]*(m+1) for _ in range(n+1)]

    for i in range(1, n+1):
        dp[i][0] = i
        back[i][0] = Op('del', i-1, 0, a[i-1], '')
    for j in range(1, m+1):
        dp[0][j] = j
        back[0][j] = Op('ins', 0, j-1, '', b[j-1])

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost_sub = 0 if a[i-1] == b[j-1] else 1

            best_cost = dp[i-1][j] + 1
            best_op = Op('del', i-1, j, a[i-1], '')

            c_ins = dp[i][j-1] + 1
            if c_ins < best_cost:
                best_cost = c_ins
                best_op = Op('ins', i, j-1, '', b[j-1])

            c_sub = dp[i-1][j-1] + cost_sub
            if c_sub < best_cost:
                best_cost = c_sub
                best_op = Op('keep' if cost_sub==0 else 'sub', i-1, j-1, a[i-1], b[j-1])

            if i >= 2 and j >= 2 and a[i-2] == b[j-1] and a[i-1] == b[j-2]:
                c_trans = dp[i-2][j-2] + 1
                if c_trans < best_cost:
                    best_cost = c_trans
                    best_op = Op('trans', i-2, j-2, a[i-2:i], b[j-2:j])

            dp[i][j] = best_cost
            back[i][j] = best_op

    # backtrace
    ops: List[Op] = []
    i, j = n, m
    while i > 0 or j > 0:
        op = back[i][j]
        if op is None:
            break
        ops.append(op)
        if op.kind == 'del':
            i -= 1
        elif op.kind == 'ins':
            j -= 1
        elif op.kind in ('sub','keep'):
            i -= 1
            j -= 1
        elif op.kind == 'trans':
            i -= 2
            j -= 2
    ops.reverse()
    return [o for o in ops if o.kind != 'keep']

def ops_to_hashcat_tokens(orig: str, trans: str, ops: List[Op]) -> List[str]:
    """Convert edit ops to hashcat rule tokens."""
    n = len(orig)
    tokens: List[str] = []

    prefix_chars: List[str] = []
    suffix_chars: List[str] = []
    mid_inserts: List[Op] = []

    for op in ops:
        if op.kind == 'ins':
            if op.i == 0:
                prefix_chars.append(op.b)
            elif op.i == n:
                suffix_chars.append(op.b)
            else:
                mid_inserts.append(op)
        elif op.kind == 'del':
            if op.i == 0:
                tokens.append('[')
            elif op.i == n-1:
                tokens.append(']')
            else:
                tokens.append(f"D{enc_pos(op.i)}")
        elif op.kind == 'sub':
            tokens.append(f"o{enc_pos(op.i)}{op.b}")
        elif op.kind == 'trans':
            i0 = op.i
            tokens.append(f"*{enc_pos(i0)}{enc_pos(i0+1)}")

    # prefix inserts: reverse so final prefix order is correct
    for ch in reversed(prefix_chars):
        tokens.insert(0, f"^{ch}")

    # suffix inserts in order
    for ch in suffix_chars:
        tokens.append(f"${ch}")

    # middle inserts by increasing pos
    for op in sorted(mid_inserts, key=lambda x: x.i):
        tokens.append(f"i{enc_pos(op.i)}{op.b}")

    return tokens

def infer_rule(orig_domain: str, trans_domain: str) -> str:
    o = sld(orig_domain)
    t = sld(trans_domain)
    ops = damerau_levenshtein_ops(o, t)
    tokens = ops_to_hashcat_tokens(o, t, ops)
    return " ".join(tokens) if tokens else ":"


# ==========================================================
# Your own (non-hashcat) rule engine for further use
# ==========================================================
OpType = str
OpArgs = Tuple[Union[int, str], ...]
Rule = List[Tuple[OpType, OpArgs]]

def parse_domain_rule_line(line: str) -> Rule:
    """
    Parse one composite rule line (space-separated tokens) into ops.
    Supported tokens: :, [, ], Dn, ^c, $c, in c, on c, *ij  (n,i,j base36)
    """
    line = line.strip()
    if not line or line.startswith("#"):
        return []

    tokens = line.split()
    ops: Rule = []
    for tok in tokens:
        if tok == ":":
            ops.append((':', ()))
        elif tok == "[":
            ops.append(('[', ()))
        elif tok == "]":
            ops.append((']', ()))
        elif tok.startswith("D") and len(tok) == 2:
            ops.append(('D', (dec_pos(tok[1]),)))
        elif tok.startswith("^") and len(tok) == 2:
            ops.append(('^', (tok[1],)))
        elif tok.startswith("$") and len(tok) == 2:
            ops.append(('$', (tok[1],)))
        elif tok.startswith("i") and len(tok) == 3:
            ops.append(('i', (dec_pos(tok[1]), tok[2])))
        elif tok.startswith("o") and len(tok) == 3:
            ops.append(('o', (dec_pos(tok[1]), tok[2])))
        elif tok.startswith("*") and len(tok) == 3:
            ops.append(('*', (dec_pos(tok[1]), dec_pos(tok[2]))))
        else:
            raise ValueError(f"Unrecognized token: {tok}")
    return ops

def apply_domain_rule(word: str, rule: Rule) -> str:
    """Apply one parsed rule to a word (left-to-right)."""
    s = word
    for op, args in rule:
        if op == ":":
            continue
        elif op == "[":
            s = s[1:] if s else s
        elif op == "]":
            s = s[:-1] if s else s
        elif op == "D":
            (pos,) = args
            pos = int(pos)
            if 0 <= pos < len(s):
                s = s[:pos] + s[pos+1:]
        elif op == "^":
            (ch,) = args
            s = str(ch) + s
        elif op == "$":
            (ch,) = args
            s = s + str(ch)
        elif op == "i":
            pos, ch = args
            pos = int(pos)
            ch = str(ch)
            if pos < 0:
                pos = 0
            if pos > len(s):
                pos = len(s)
            s = s[:pos] + ch + s[pos:]
        elif op == "o":
            pos, ch = args
            pos = int(pos)
            ch = str(ch)
            if 0 <= pos < len(s):
                s = s[:pos] + ch + s[pos+1:]
        elif op == "*":
            i, j = map(int, args)
            if 0 <= i < len(s) and 0 <= j < len(s) and i != j:
                lst = list(s)
                lst[i], lst[j] = lst[j], lst[i]
                s = "".join(lst)
        else:
            raise RuntimeError(f"Unknown op: {op}")
    return s

def load_sorted_domain_rules(path: str = "./domain.rule") -> List[Rule]:
    """
    Load domain.rule (already sorted by frequency) and parse into our Rule objects.
    """
    rules: List[Rule] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            rules.append(parse_domain_rule_line(line))
    return rules

def apply_domain_rules(word: str, rules: Iterable[Rule]) -> Set[str]:
    """Apply many rules and return all outputs."""
    out: Set[str] = set()
    for r in rules:
        try:
            out.add(apply_domain_rule(word, r))
        except Exception:
            pass
    return out

In [39]:
with open('./disputes-training.json', "r", encoding="utf-8") as f:
    data = json.load(f)

counts = Counter()
for orig, trans_map in data.items():
    for trans in trans_map.keys():
        rule = infer_rule(orig, trans)
        counts[rule] += 1

with open('./rule_counts.json', "w", encoding="utf-8") as f:
    json.dump(counts, f, indent=2, sort_keys=True)

# sort by usage desc, stable tie-break
sorted_rules = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))

with open('./domain.rule', "w", encoding="utf-8") as f:
    for r, c in sorted_rules:
        f.write(r + "\n")


In [40]:
class Company:
    def __init__(
            self, name, domain, sector, market_value, 
            security_analysts, financial_assets, 
            domain_popularity, osint_domains
        ):
        self.name = name
        self.domain = domain
        self.sector = sector
        self.market_value = market_value
        self.security_analysts = security_analysts
        self.financial_assets = financial_assets
        self.domain_popularity = domain_popularity
        self.osint_domains = osint_domains

class CompanyLoader:
    def __init__(self, infile: str):
        self.companies = {}
        with open(infile, 'rt') as fin:
            for line in fin:
                data = json.loads(line)
                self.companies[data['domain']] = Company(
                    name = data['company'],
                    domain = data['domain'],
                    sector = data['naics_sector'],
                    market_value = data['market_value'],
                    security_analysts = data['security_analysts_est'],
                    financial_assets = data['assets'],
                    domain_popularity = data['popularity'],
                    osint_domains = data['osint_domains']
                )
    def get(self, name) -> Company:
        return self.companies.get(name)

In [41]:
class BrandProtector:
    def __init__(self, zone_infile: str, companies_infile: str, disputes_infile: str = None):
        # set the cost of the various actions
        self.dispute_cost = 1200
        self.registration_cost = 200
        self.budget = 100000

        self.qwerty = {
            '1': '2q', '2': '3wq1', '3': '4ew2', '4': '5re3', '5': '6tr4', '6': '7yt5', '7': '8uy6', '8': '9iu7', '9': '0oi8', '0': 'po9',
            'q': '12wa', 'w': '3esaq2', 'e': '4rdsw3', 'r': '5tfde4', 't': '6ygfr5', 'y': '7uhgt6', 'u': '8ijhy7', 'i': '9okju8', 'o': '0plki9', 'p': 'lo0',
            'a': 'qwsz', 's': 'edxzaw', 'd': 'rfcxse', 'f': 'tgvcdr', 'g': 'yhbvft', 'h': 'ujnbgy', 'j': 'ikmnhu', 'k': 'olmji', 'l': 'kop',
            'z': 'asx', 'x': 'zsdc', 'c': 'xdfv', 'v': 'cfgb', 'b': 'vghn', 'n': 'bhjm', 'm': 'njk', '-': '0p'
            }

        with open(zone_infile, 'rt') as fin:
            self.registered_domains = {parts[0][:-1] for line in fin if (parts := line.split())}
        self.company_loader = CompanyLoader(companies_infile)

        if disputes_infile:
            with open(disputes_infile, 'rt') as fin:
                self.disputes = json.load(fin)
        else:
            self.disputes = {}

    def generate(self, domain: str):
        # verify it is a valid domain
        parts = tldextract.extract(domain.lower())
        e2ld, tld = parts.domain, parts.suffix

        domains = set()
        if e2ld and tld:
            # the five models of Wang et al., “Strider typo-patrol: Discovery and analysis of systematic typo-squatting,” 2nd Workshop on Steps to Reducing Unwanted Traffic on the Internet, 2006
            ## missing dot
            domains.add(f'www{e2ld}.{tld}')
            # character insertion and replacement
            for idx, orig in enumerate(e2ld):
                # insert before the character
                domains.update([f'{e2ld[:idx]}{ins}{e2ld[idx:]}.{tld}' for ins in self.qwerty[orig] + orig])
                # insert after the character
                domains.update([f'{e2ld[:idx + 1]}{ins}{e2ld[idx + 1:]}.{tld}' for ins in self.qwerty[orig] + orig])
                
                # replace the character
                domains.update([f'{e2ld[:idx]}{repl}{e2ld[idx + 1:]}.{tld}' for repl in self.qwerty[orig] + orig])
            
            # character transposition
            domains.update([
                f"{e2ld[:l]}{e2ld[r]}{e2ld[l]}{e2ld[r+1:]}.{tld}" 
                for l, r in zip(
                    range(len(e2ld)), range(1, len(e2ld))
                ) if e2ld[r] != e2ld[l]
            ])
        # registered domains are not considered
        return domains.difference(self.registered_domains)


    #TODO: Implement the function `predict`
    def predict(self, outfile: str):
        # get company's details
        candidates = set()
        rules = load_sorted_domain_rules('./domain.rule')
        budget_to_spend = int(0.8 * self.budget)
        num_registrations = int(budget_to_spend // self.registration_cost)
        num_per_company = max(1, num_registrations // len(self.company_loader.companies))
        # for each company, generate the transformations by the first 15 rules
        for domain, _ in self.company_loader.companies.items():
            for rule in rules[:num_per_company]:
                transformed_domains = apply_domain_rules(domain, [rule])
                for transformed_domain in transformed_domains:
                    if transformed_domain in self.registered_domains:
                        continue
                    candidates.add((domain, transformed_domain))
        
        to_register = list(candidates)[:num_registrations]
        
        # write to the output file
        with open(outfile, 'wt') as fout:
            fout.write('\n'.join([f'{base}:{mod}' for base, mod in to_register]))
    
    # **DO NOT MODIFY**
    def score(self, infile: str):
        if self.disputes:
            disputes = copy.deepcopy(self.disputes)
            profits = self.budget
            correct_guesses = Counter()
            missed_guesses = Counter()

            registrations = 0

            with open(infile, 'r') as submission_file:
                while profits:
                    line = submission_file.readline()
                    if line.strip():
                        root, domain = line.strip().split(':')
                        profits -= self.registration_cost

                        if root in disputes and domain in disputes[root]:
                            correct_guesses.update([disputes[root].pop(domain)])

                        registrations += 1
                    elif not submission_file.read():
                        break

            # estimate the penalties
            penalties = 0
            for root, entries in disputes.items():
                missed_guesses.update(entries.values())

                for _, risk_level in entries.items():
                    penalties += risk_level * 0.25 * self.dispute_cost
            profits -= penalties

            return {
                'domains_registered': registrations,
                'disputes_avoided': {
                    'low-risk': correct_guesses[1],
                    'medium-risk': correct_guesses[2],
                    'high-risk': correct_guesses[3]
                },
                'disputes_encountered': {
                    'low-risk': missed_guesses[1],
                    'medium-risk': missed_guesses[2],
                    'high-risk': missed_guesses[3]
                },
                'registration_cost': registrations * self.registration_cost,
                'penalties': penalties,
                'profits': max(0, profits)
            }
        else:
            print('Disputes data not available. Performance cannot be evaluated.')

In [42]:
zonefile = 'zonefile.txt'
# for the training data
companies_infile = 'companies-training.jsonl'
disputes_infile = 'disputes-training.json'
obp = BrandProtector(zonefile, companies_infile, disputes_infile)
# or this for the validation data (used on the leaderboard)
# companies_infile = 'companies-validation.jsonl'
# obp = BrandProtector(zonefile, companies_infile)

# run the prediction code and save it to submission.txt
obp.predict('submission.txt')

# get the evaluation scores
obp.score('submission.txt')

{'domains_registered': 297,
 'disputes_avoided': {'low-risk': 99, 'medium-risk': 11, 'high-risk': 0},
 'disputes_encountered': {'low-risk': 407, 'medium-risk': 69, 'high-risk': 5},
 'registration_cost': 59400,
 'penalties': 168000.0,
 'profits': 0}